# CXR-DetectBench Phase 2 - Data Preprocessing

Execute Phase 2 preprocessing pipeline:
- Multi-annotator fusion (raw / wbf / nms)
- Fusion ablation with YOLOv8n
- COCO to YOLO format conversion

## 1. Environment Setup

In [ ]:
!pip install -q ensemble-boxes pycocotools opencv-python-headless ultralytics scikit-learn tqdm matplotlib

In [ ]:
import os
from pathlib import Path

os.chdir('/kaggle/working')

print("Input datasets:")
!ls -la /kaggle/input/

## 2. Clone Repository

In [ ]:
!git clone https://github.com/kinjazA/cxr-detectbench.git
%cd cxr-detectbench

## 3. Prepare Data

In [ ]:
!mkdir -p data/raw data/processed

# Copy metadata
!cp /kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train.csv data/raw/
!cp /kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train_meta.csv data/raw/images.csv

# Symlink images (skip CLAHE for speed, use original PNGs)
!ln -s /kaggle/input/vinbigdata-chest-xray-original-png/train data/processed/images_png

!ls -lh data/raw/
!ls -lh data/processed/

## 4. Multi-Annotator Fusion

In [ ]:
# Generate 3 COCO annotation variants: raw / wbf / nms
!python scripts/label_fusion.py \
    --train_csv data/raw/train.csv \
    --images_csv data/raw/images.csv \
    --output_dir data/processed/labels_coco \
    --iou_thr 0.5

In [ ]:
# Check outputs
!python scripts/check_fusion_output.py

## 5. Fusion Ablation Experiment

In [ ]:
# Quick YOLOv8n training on 3 fusion variants (20 epochs)
!python scripts/run_fusion_ablation.py \
    --coco_dir data/processed/labels_coco \
    --images_dir data/processed/images_png \
    --train_csv data/raw/train.csv \
    --epochs 20 \
    --imgsz 640 \
    --batch 16 \
    --output phase2_fusion_ablation.csv

In [ ]:
# Display results
import pandas as pd

results = pd.read_csv('phase2_fusion_ablation.csv')
print("\n" + "="*60)
print("Phase 2.4 Fusion Strategy Ablation Results")
print("="*60)
print(results.to_string(index=False))

best_mode = results.loc[results['mAP50'].idxmax(), 'fusion_mode']
print(f"\nBest fusion strategy: {best_mode}")

## 6. Generate Final YOLO Labels

In [ ]:
# Convert best fusion to YOLO format
# Manually set BEST_FUSION based on above results
BEST_FUSION = 'wbf'  # Update this after checking ablation results

!python scripts/convert_coco_yolo.py \
    --coco_json data/processed/labels_coco/{BEST_FUSION}/annotations.json \
    --output_dir data/processed/labels_yolo/{BEST_FUSION}

## 7. Verify Outputs

In [ ]:
print("=== Phase 2 Outputs ===")
print("\n1. COCO annotations:")
!ls -lh data/processed/labels_coco/*/annotations.json

print("\n2. YOLO labels:")
!ls data/processed/labels_yolo/{BEST_FUSION}/ | head -20

print("\n3. Ablation results:")
!cat phase2_fusion_ablation.csv

print("\n4. Image count:")
!ls data/processed/images_png/ | wc -l

print("\n✅ Phase 2 complete! Save Version to persist outputs.")